# 00 — GBIF Occurrence Data

Download and filter Swiss plant occurrences from the GBIF open data snapshot on S3.

In [ ]:
import pyarrow.parquet as pq
import s3fs

s3 = s3fs.S3FileSystem(anon=True)

example_file = "s3://gbif-open-data-eu-central-1/occurrence/2024-03-01/occurrence.parquet/000001"

with s3.open(example_file) as f:
    schema = pq.read_schema(f)
    column_names = schema.names

print(f"The dataset has {len(column_names)} columns.")
print("Columns:", column_names)

In [ ]:
import duckdb

S3_BUCKET_PATH = "s3://gbif-open-data-eu-central-1/occurrence/2026-04-01/occurrence.parquet/*"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='eu-central-1';")

query = f"""
COPY (
  SELECT
    gbifid, species, specieskey, decimallatitude, decimallongitude,
    coordinateuncertaintyinmeters, eventdate, occurrencestatus
  FROM read_parquet('{S3_BUCKET_PATH}')
  WHERE countrycode = 'CH'
    AND occurrencestatus = 'PRESENT'
    AND decimallatitude IS NOT NULL
    AND kingdom = 'Plantae'
) TO 'swiss_occurences.parquet' (FORMAT PARQUET);
"""
con.execute(query)